# Sovereign bond futures from EMIR

Builds `lab_prj_emir_ecb.hermesf_fut` (raw) and `lab_prj_emir_ecb.hermesf_fut_clean` (cleaned) from the ECB EMIR trade state tables.
Euro area contracts (Eurex, ISIN list in `gov_fut.csv`) and US Treasury futures (CBOT, identified by venue and CFI).
One row per trade state and reporting day, with buyer and seller instead of reporter and direction.

Steps: parameters, query builders, one day tests, raw tables year by year, reference price, cleaned table, checks.

In [ ]:
import pyodbc
import pandas as pd
import time
from datetime import datetime

cnxn = pyodbc.connect('DSN=Hermes_DSN', autocommit=True)
cursor = cnxn.cursor()

In [ ]:
# parameters
eurex_lei = ('529900LN3S50JPU47S06', '529900VIIHQECIS4ET47', '529900UT4DG0LG5R9O07')

prerefit_start, prerefit_end = '2018-01-01', '2024-04-26'
refit_start,    refit_end    = '2025-01-01', '2026-09-01'   # REFIT transition window (Apr to Dec 2024) excluded

schema = 'lab_prj_emir_ecb'                                 # where the output tables go
sector_table = 'xlab_ecb_prj_sftds_cb_common.pth_sector'

In [ ]:
# euro area contract list (ISIN, country, expiration date) as an inline SQL table
euro_govs = pd.read_csv('gov_fut.csv')
gov_fut_sql = '\n    UNION ALL '.join(
    f"SELECT '{r.isin}' AS isin, '{r.country}' AS country, '{r.expiration_date}' AS expiration_date"
    for r in euro_govs.itertuples()
)

## Query builders, one per regime and region

In [ ]:
# pre-REFIT EA query (Eurex sovereign futures)
def prerefit_query(start, end):
    return f"""
SELECT
    -- identifiers
    tec_ruti,
    reference_period,
    -- counterparties
    CASE WHEN direction = 'BYER' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS buyer_id,
    CASE WHEN direction = 'BYER' THEN rep_sector             ELSE oth_sector             END AS buyer_sector,
    CASE WHEN direction = 'BYER' THEN rep_country            ELSE oth_country            END AS buyer_country,
    CASE WHEN direction = 'SLLR' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS seller_id,
    CASE WHEN direction = 'SLLR' THEN rep_sector             ELSE oth_sector             END AS seller_sector,
    CASE WHEN direction = 'SLLR' THEN rep_country            ELSE oth_country            END AS seller_country,
    -- product
    product_id,
    product_country,
    expiration_date,
    -- trade terms
    notional,
    notional_currency,
    price,
    quantity
FROM (
    SELECT
        -- identifiers
        e.tec_ruti,
        e.reference_period,
        -- counterparties
        e.leg1_counterparty_side AS direction,
        e.leg1_reporting_cpty_id,
        e.leg1_other_cpty_id,
        e.leg1_reporting_cpty_id_country_code AS rep_country,
        e.leg1_other_cpty_id_country_code     AS oth_country,
        CASE WHEN e.leg1_reporting_cpty_id IN {eurex_lei} THEN 'CCP' ELSE s_rep.sector END AS rep_sector,
        CASE WHEN e.leg1_other_cpty_id     IN {eurex_lei} THEN 'CCP' ELSE s_oth.sector END AS oth_sector,
        -- product
        CAST(e.leg1_product_id AS STRING)    AS product_id,
        g.country          AS product_country,
        g.expiration_date  AS expiration_date,
        -- trade terms
        e.leg1_notional            AS notional,
        e.leg1_notional_currency1  AS notional_currency,
        e.leg1_price_rate          AS price,
        e.leg1_quantity            AS quantity
    FROM crp_emir_ecb.emir_ecb_states_deduplicated e
    JOIN (
        {gov_fut_sql}
    ) g
        ON e.leg1_product_id = g.isin
    LEFT JOIN {sector_table} s_rep
        ON e.leg1_reporting_cpty_id = s_rep.lei
    LEFT JOIN {sector_table} s_oth
        ON e.leg1_other_cpty_id = s_oth.lei
    WHERE e.reference_period BETWEEN '{start}' AND '{end}'
        AND e.leg1_counterparty_side IN ('BYER', 'SLLR')
        AND e.leg1_notional > 0
        AND e.leg1_notional <= 1e11
) base
"""

In [ ]:
# REFIT EA query (Eurex sovereign futures)
def refit_query(start, end):
    return f"""
SELECT
    -- identifiers
    tec_ruti,
    reference_period,
    -- counterparties
    CASE WHEN direction = 'BYER' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS buyer_id,
    CASE WHEN direction = 'BYER' THEN rep_sector             ELSE oth_sector             END AS buyer_sector,
    CASE WHEN direction = 'BYER' THEN rep_country            ELSE oth_country            END AS buyer_country,
    CASE WHEN direction = 'SLLR' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS seller_id,
    CASE WHEN direction = 'SLLR' THEN rep_sector             ELSE oth_sector             END AS seller_sector,
    CASE WHEN direction = 'SLLR' THEN rep_country            ELSE oth_country            END AS seller_country,
    -- product
    product_id,
    product_country,
    expiration_date,
    -- trade terms
    notional,
    notional_currency,
    price,
    quantity
FROM (
    SELECT
        -- identifiers
        e.tec_ruti,
        e.reference_period,
        -- counterparties
        e.leg1_direction AS direction,
        e.leg1_reporting_cpty_id,
        e.leg1_other_cpty_id,
        e.leg1_reporting_cpty_id_country_code_gleif AS rep_country,
        e.leg1_other_cpty_id_country_code_gleif     AS oth_country,
        CASE WHEN e.leg1_reporting_cpty_id IN {eurex_lei} THEN 'CCP' ELSE s_rep.sector END AS rep_sector,
        CASE WHEN e.leg1_other_cpty_id     IN {eurex_lei} THEN 'CCP' ELSE s_oth.sector END AS oth_sector,
        -- product
        CAST(e.leg1_product_isin AS STRING)  AS product_id,
        g.country            AS product_country,
        g.expiration_date    AS expiration_date,
        -- trade terms
        e.leg1_notional_leg1           AS notional,
        e.leg1_notional_leg1_currency  AS notional_currency,
        e.leg1_price                   AS price,
        e.leg1_notional_quantity_leg1  AS quantity
    FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_deduplicated e
    JOIN (
        {gov_fut_sql}
    ) g
        ON e.leg1_product_isin = g.isin
    LEFT JOIN {sector_table} s_rep
        ON e.leg1_reporting_cpty_id = s_rep.lei
    LEFT JOIN {sector_table} s_oth
        ON e.leg1_other_cpty_id = s_oth.lei
    WHERE e.reference_period BETWEEN '{start}' AND '{end}'
        AND e.leg1_direction IN ('BYER', 'SLLR')
        AND e.leg1_notional_leg1 > 0
        AND e.leg1_notional_leg1 <= 1e11
) base
"""

In [ ]:
# pre-REFIT US query (CBOT Treasury futures)
# no contract ISIN on CBOT, so product_id is synthetic: venue, CFI and expiry
def prerefit_us_query(start, end):
    return f"""
SELECT
    -- identifiers
    tec_ruti,
    reference_period,
    -- counterparties
    CASE WHEN direction = 'BYER' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS buyer_id,
    CASE WHEN direction = 'BYER' THEN rep_sector             ELSE oth_sector             END AS buyer_sector,
    CASE WHEN direction = 'BYER' THEN rep_country            ELSE oth_country            END AS buyer_country,
    CASE WHEN direction = 'SLLR' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS seller_id,
    CASE WHEN direction = 'SLLR' THEN rep_sector             ELSE oth_sector             END AS seller_sector,
    CASE WHEN direction = 'SLLR' THEN rep_country            ELSE oth_country            END AS seller_country,
    -- product
    product_id,
    product_country,
    expiration_date,
    -- trade terms
    notional,
    notional_currency,
    price,
    quantity
FROM (
    SELECT
        -- identifiers
        e.tec_ruti,
        e.reference_period,
        -- counterparties
        e.leg1_counterparty_side AS direction,
        e.leg1_reporting_cpty_id,
        e.leg1_other_cpty_id,
        e.leg1_reporting_cpty_id_country_code AS rep_country,
        e.leg1_other_cpty_id_country_code     AS oth_country,
        CASE WHEN e.leg1_reporting_cpty_id IN {eurex_lei} THEN 'CCP' ELSE s_rep.sector END AS rep_sector,
        CASE WHEN e.leg1_other_cpty_id     IN {eurex_lei} THEN 'CCP' ELSE s_oth.sector END AS oth_sector,
        -- product
        CONCAT('XCBT_FFDPSX_', to_date(e.leg1_maturity_date)) AS product_id,
        'US'                                                 AS product_country,
        to_date(e.leg1_maturity_date)                        AS expiration_date,
        -- trade terms
        e.leg1_notional            AS notional,
        e.leg1_notional_currency1  AS notional_currency,
        e.leg1_price_rate          AS price,
        e.leg1_quantity            AS quantity
    FROM crp_emir_ecb.emir_ecb_states_deduplicated e
    LEFT JOIN {sector_table} s_rep
        ON e.leg1_reporting_cpty_id = s_rep.lei
    LEFT JOIN {sector_table} s_oth
        ON e.leg1_other_cpty_id = s_oth.lei
    WHERE e.reference_period BETWEEN '{start}' AND '{end}'
        AND e.leg1_execution_venue = 'XCBT'
        AND e.leg1_contract_type = 'FUTR'
        AND e.leg1_asset_class = 'INTR'
        AND e.leg1_product_clssfctn = 'FFDPSX'
        AND e.leg1_notional_currency1 = 'USD'
        AND e.leg1_maturity_date IS NOT NULL
        AND e.leg1_counterparty_side IN ('BYER', 'SLLR')
        AND e.leg1_notional > 0
        AND e.leg1_notional <= 1e11
) base
"""

In [ ]:
# REFIT US query (CBOT Treasury futures)
# no contract ISIN on CBOT, so product_id is synthetic: venue, CFI and expiry
def refit_us_query(start, end):
    return f"""
SELECT
    -- identifiers
    tec_ruti,
    reference_period,
    -- counterparties
    CASE WHEN direction = 'BYER' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS buyer_id,
    CASE WHEN direction = 'BYER' THEN rep_sector             ELSE oth_sector             END AS buyer_sector,
    CASE WHEN direction = 'BYER' THEN rep_country            ELSE oth_country            END AS buyer_country,
    CASE WHEN direction = 'SLLR' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS seller_id,
    CASE WHEN direction = 'SLLR' THEN rep_sector             ELSE oth_sector             END AS seller_sector,
    CASE WHEN direction = 'SLLR' THEN rep_country            ELSE oth_country            END AS seller_country,
    -- product
    product_id,
    product_country,
    expiration_date,
    -- trade terms
    notional,
    notional_currency,
    price,
    quantity
FROM (
    SELECT
        -- identifiers
        e.tec_ruti,
        e.reference_period,
        -- counterparties
        e.leg1_direction AS direction,
        e.leg1_reporting_cpty_id,
        e.leg1_other_cpty_id,
        e.leg1_reporting_cpty_id_country_code_gleif AS rep_country,
        e.leg1_other_cpty_id_country_code_gleif     AS oth_country,
        CASE WHEN e.leg1_reporting_cpty_id IN {eurex_lei} THEN 'CCP' ELSE s_rep.sector END AS rep_sector,
        CASE WHEN e.leg1_other_cpty_id     IN {eurex_lei} THEN 'CCP' ELSE s_oth.sector END AS oth_sector,
        -- product
        CONCAT('XCBT_FFDPSX_', to_date(e.leg1_maturity_date)) AS product_id,
        'US'                                                 AS product_country,
        to_date(e.leg1_maturity_date)                        AS expiration_date,
        -- trade terms
        e.leg1_notional_leg1           AS notional,
        e.leg1_notional_leg1_currency  AS notional_currency,
        e.leg1_price                   AS price,
        e.leg1_notional_quantity_leg1  AS quantity
    FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_deduplicated e
    LEFT JOIN {sector_table} s_rep
        ON e.leg1_reporting_cpty_id = s_rep.lei
    LEFT JOIN {sector_table} s_oth
        ON e.leg1_other_cpty_id = s_oth.lei
    WHERE e.reference_period BETWEEN '{start}' AND '{end}'
        AND e.leg1_execution_venue = 'XCBT'
        AND e.leg1_contract_type = 'FUTR'
        AND e.leg1_asset_class = 'INTR'
        AND e.leg1_product_cfi = 'FFDPSX'
        AND e.leg1_notional_leg1_currency = 'USD'
        AND e.leg1_maturity_date IS NOT NULL
        AND e.leg1_direction IN ('BYER', 'SLLR')
        AND e.leg1_notional_leg1 > 0
        AND e.leg1_notional_leg1 <= 1e11
) base
"""

## One day tests

In [ ]:
# test: one day per regime and region
tests = {
    'prerefit_ea': prerefit_query('2023-07-17', '2023-07-17'),
    'prerefit_us': prerefit_us_query('2023-07-17', '2023-07-17'),
    'refit_ea':    refit_query('2024-07-15', '2024-07-15'),
    'refit_us':    refit_us_query('2024-07-15', '2024-07-15'),
}
dfs = {k: pd.read_sql_query(q, cnxn) for k, q in tests.items()}
for k, d in dfs.items():
    print(k, d.shape, list(d.columns) == list(dfs['refit_ea'].columns))
dfs['refit_us'].head()

## Raw tables, built year by year

In [ ]:
# build a table in yearly chunks: CREATE from the first chunk, INSERT INTO for the rest
import time
from datetime import datetime

def year_chunks(start, end):
    chunks, y = [], int(start[:4])
    while True:
        a = max(start, f'{y}-01-01')
        b = min(end, f'{y}-12-31')
        chunks.append((a, b))
        if b == end:
            return chunks
        y += 1

def build_table(table, queries, start, end):
    first = True
    for a, b in year_chunks(start, end):
        for region, q in queries.items():
            t0 = time.time()
            if first:
                cursor.execute(f"DROP TABLE IF EXISTS {table}")
                cursor.execute(f"""
CREATE EXTERNAL TABLE {table}
STORED AS PARQUET TBLPROPERTIES ('external.table.purge'='true')
AS
{q(a, b)}
""")
                first = False
            else:
                cursor.execute(f"INSERT INTO {table}\n{q(a, b)}")
            print(f"{datetime.now():%H:%M:%S}  {table.split('.')[-1]:22s} {region}  {a} to {b}  done in {(time.time() - t0) / 60:5.1f} min")

In [ ]:
# create pre-REFIT table (EA and US), year by year
build_table(f'{schema}.hermesf_fut_prerefit',
            {'EA': prerefit_query, 'US': prerefit_us_query},
            prerefit_start, prerefit_end)

In [ ]:
# create REFIT table (EA and US), year by year
build_table(f'{schema}.hermesf_fut_refit',
            {'EA': refit_query, 'US': refit_us_query},
            refit_start, refit_end)

In [ ]:
# full series: pre-REFIT and REFIT stacked (date windows from the parameters)
# numeric columns cast to DOUBLE, the two source tables have different decimal scales
cols = """tec_ruti, reference_period,
       buyer_id, buyer_sector, buyer_country, seller_id, seller_sector, seller_country,
       product_id, product_country, expiration_date,
       CAST(notional AS DOUBLE) AS notional, notional_currency,
       CAST(price AS DOUBLE) AS price, CAST(quantity AS DOUBLE) AS quantity"""
cursor.execute(f"DROP TABLE IF EXISTS {schema}.hermesf_fut")
cursor.execute(f"""
CREATE EXTERNAL TABLE {schema}.hermesf_fut
STORED AS PARQUET TBLPROPERTIES ('external.table.purge'='true')
AS
SELECT {cols} FROM {schema}.hermesf_fut_prerefit
UNION ALL
SELECT {cols} FROM {schema}.hermesf_fut_refit
""")

In [ ]:
# check
pd.read_sql_query(f"""
SELECT product_country, MIN(reference_period) AS first_date, MAX(reference_period) AS last_date,
       COUNT(*) AS n_rows, COUNT(DISTINCT product_id) AS n_contracts
FROM {schema}.hermesf_fut
GROUP BY product_country
ORDER BY product_country
""", cnxn)

## Cleaned table

Rules, applied on top of the raw table.
1. Expiry cut off: rows reported more than 10 days after the contract's expiration date are dropped.
2. Price normalised to a per 100 quotation and only kept if between 50 and 250. Where missing, the reference price is used, the average normalised price of all reporters for the same contract and day.
3. Notional rebuild: where notional / (quantity x price) > 3000 the notional is replaced by quantity x 1000 x price, quantity treated as contracts.
4. Cap: rows with more than 500,000 implied contracts (notional / (price x 1000)) are dropped.

The raw notional and price are kept as `notional_raw` and `price_raw`.

In [ ]:
# cleaning step 1: reference price per contract and day, built year by year
# price normalised to per 100 and only kept if between 50 and 250, then the average across all reporters
def price_ref_query(start, end):
    return f"""
SELECT product_id, reference_period, AVG(price_norm) AS price_ref, COUNT(*) AS n_prices
FROM (
    SELECT product_id, reference_period,
           CASE WHEN price > 25000 THEN price / 1000
                WHEN price > 2500  THEN price / 100
                WHEN price > 250   THEN price / 10
                ELSE price END AS price_norm
    FROM {schema}.hermesf_fut
    WHERE reference_period BETWEEN '{start}' AND '{end}'
) a
WHERE price_norm BETWEEN 50 AND 250
GROUP BY product_id, reference_period
"""

build_table(f'{schema}.hermesf_fut_price_ref', {'ALL': price_ref_query}, prerefit_start, refit_end)

In [ ]:
# cleaning step 2: the cleaned table, built year by year on top of the raw one
def clean_query(start, end):
    return f"""
SELECT *
FROM (
    SELECT r.tec_ruti, r.reference_period,
           r.buyer_id, r.buyer_sector, r.buyer_country, r.seller_id, r.seller_sector, r.seller_country,
           r.product_id, r.product_country, r.expiration_date,
           r.notional AS notional_raw, r.notional_currency, r.price AS price_raw, r.quantity,
           COALESCE(r.price_norm, p.price_ref) AS price_clean,
           CASE WHEN r.quantity IS NULL OR r.quantity <= 0 OR COALESCE(r.price_norm, p.price_ref) IS NULL THEN r.notional
                WHEN r.notional / (r.quantity * COALESCE(r.price_norm, p.price_ref)) > 3000
                     THEN r.quantity * 1000 * COALESCE(r.price_norm, p.price_ref)
                ELSE r.notional END AS notional
    FROM (
        SELECT *, CASE WHEN pn BETWEEN 50 AND 250 THEN pn END AS price_norm
        FROM (
            SELECT *,
                   CASE WHEN price > 25000 THEN price / 1000
                        WHEN price > 2500  THEN price / 100
                        WHEN price > 250   THEN price / 10
                        ELSE price END AS pn
            FROM {schema}.hermesf_fut
            WHERE reference_period BETWEEN '{start}' AND '{end}'
                AND reference_period <= to_date(date_add(CAST(expiration_date AS TIMESTAMP), 10))
        ) a
    ) r
    LEFT JOIN {schema}.hermesf_fut_price_ref p
        ON r.product_id = p.product_id AND r.reference_period = p.reference_period
) c
WHERE price_clean IS NULL OR notional / (price_clean * 1000) <= 500000
"""

build_table(f'{schema}.hermesf_fut_clean', {'ALL': clean_query}, prerefit_start, refit_end)

In [ ]:
# check
pd.read_sql_query(f"""
SELECT product_country, MIN(reference_period) AS first_date, MAX(reference_period) AS last_date,
       COUNT(*) AS n_rows, COUNT(DISTINCT product_id) AS n_contracts,
       ROUND(SUM(notional_raw) / 1e12, 1) AS notional_raw_tn, ROUND(SUM(notional) / 1e12, 1) AS notional_clean_tn
FROM {schema}.hermesf_fut_clean
GROUP BY product_country
ORDER BY product_country
""", cnxn)

## Net positions by sector

In [ ]:
# net positions, cleaned
query_clean = f"""
SELECT reference_period AS bdate, product_country, sector,
       SUM(net) / 1e9 AS net_position_bn
FROM (
    SELECT reference_period, product_country, buyer_sector  AS sector,  notional AS net FROM {schema}.hermesf_fut_clean
    UNION ALL
    SELECT reference_period, product_country, seller_sector AS sector, -notional AS net FROM {schema}.hermesf_fut_clean
) x
GROUP BY reference_period, product_country, sector
ORDER BY reference_period, product_country, sector
"""
df_net_clean = pd.read_sql_query(query_clean, cnxn)

for country, d in df_net_clean.groupby('product_country'):
    d.pivot(index='bdate', columns='sector', values='net_position_bn').plot(title=f'{country} (cleaned)', figsize=(12, 5))